# Alkaline platform 270

Every cell here moves a real arm. Two things that are true no matter what the code says:

- **The physical e-stop is the only reliable stop.** The voice stop in the RobotStop
  repo is measured at 42% recall; the software envelope below does not constrain
  joint-space moves at all.
- **`home()` is a joint-space move and nothing checks it.** Its angles were last set
  for a robot swap and drove the arm into the table on 2026-09-16. Raise the arm to a
  clear pose before calling it, and see the warning at `ang_dict` in
  `robotic_testing/common/robotic_arms/xarm7/xarm7_config.py`.

The same arm control is available as a command line tool, which needs no notebook
server: `python robotic_testing/vla.py --help`

## Session

Arm-only session via `init_vla_platform` — no EC-Lab, camera, or TeamViewer/ToDesk
bypasses, none of which the immersion-sweep cells below need. If you need the full
platform (EC-Lab, camera, database) to run `rt.run(...)` electrochemistry, swap this
cell back to `init_alkaline_platform_270`, which needs all of it.

In [ ]:
from robotic_testing.platforms import init_vla_platform

exp_name = 'test'
rt = init_vla_platform(exp_name=exp_name, connect_db=False)
arm = rt.arm

In [ ]:
# start in n hours instead of now
import time

n = 0.1
time.sleep(3600 * n)

## Arm basics

In [ ]:
# Uncomment the one you want. Each is a complete move on its own.

# arm.home()
# arm.move_to_mid_station()
# arm.open_gripper()
# arm.close_gripper()

# where is it now, and is that inside the safety envelope?
# print(arm.describe_state())
# print(arm.describe_envelope())

## Samples

In [ ]:
sample = 0

arm.load_sample(sample)
# arm.unload_sample(sample)

# arm.pick_up_sample(sample)
# arm.put_sample_back(sample)

# a rack position that needs the offset correction applied
# arm.pick_up_sample(11, correction=True)

# a whole rack, one at a time
# for i in range(30, 36):
#     arm.pick_up_sample(i)
#     arm.put_sample_back(i)

## Flask and rinsing

In [ ]:
arm.move_to_flask()

# in, then back out
# arm.sink_in_flask()
# arm.sink_in_flask(reverse=True)

# arm.rinsing()

## Safe immersion sweep, with per-sample dip/hold/lift

Same waypoints as `load_sample`/`unload_sample` — pick up, `mid_station`, above the flask, immersed, `mid_station`, put back — nothing skipped. The final descent and lift segments run at a speed you choose, instead of `sink_in_flask`'s fixed `slow_3` (~12.5 mm/s), and each sample in `SAMPLES` can have its own dip speed, hold time, and lift speed via `per_sample_settings`. Sample rack positions and the flask position are untouched — this only changes speed/timing, never where anything moves to.

Why this is safe: last time's self-collision came from *skipping* `mid_station`, which changed the geometric path enough that the controller picked a different, colliding joint solution. Speed and hold time don't change the path or target pose at all, so they don't carry that risk.

`sink_in_flask_controlled_speed` isn't used here — it looks up `megnan_<speed>` entries in `pos_settings_dict` that only exist for xArm6, not xArm7 (what this platform uses). Speed is set directly instead.

**First real run with a new sample list: try just one or two samples first, hand on the e-stop, before running the full set.**

In [ ]:
# Same waypoints as load_sample/unload_sample (including mid_station), minus rinsing --
# only the final descend/lift segment runs at a chosen speed instead of sink_in_flask's
# fixed slow_3. Sample rack positions and the flask position are untouched throughout --
# only speed and hold time vary per sample.
import time

SAMPLES = [0]                   # sample slots to run, in order. e.g. [0, 1, 2]
DEFAULT_DIP_SPEED_MM_S = 20.0   # descent (immersion) speed, mm/s, for samples not in per_sample_settings
DEFAULT_LIFT_SPEED_MM_S = 20.0  # lift (withdrawal) speed, mm/s, for samples not in per_sample_settings
DEFAULT_HOLD_S = 1              # seconds to hold immersed, 0 = no hold, for samples not in per_sample_settings

# Per-sample overrides: {sample index: [dip speed mm/s, hold seconds, lift speed mm/s]}
# Any sample in SAMPLES that isn't listed here uses the DEFAULT_* values above.
per_sample_settings = {
    # 0: [5.0, 10, 2.0],
}


def sample_settings(sample):
    dip, hold, lift = per_sample_settings.get(
        sample, [DEFAULT_DIP_SPEED_MM_S, DEFAULT_HOLD_S, DEFAULT_LIFT_SPEED_MM_S]
    )
    return dip, hold, lift


cfg = arm.config


def at_speed(mm_s):
    """Default motion settings with the speed swapped to mm_s."""
    if not 0 < mm_s <= 100:
        raise ValueError(f'Speed {mm_s} mm/s must be between 0 and 100')
    return {**cfg.pos_settings_dict['default'], 'speed': mm_s}


def load_sample_at_speed(sample, dip_speed):
    # Same steps as arm.load_sample, just with a chosen speed for the descent.
    arm.pick_up_sample(sample)
    arm.move_to_mid_station()
    arm.move_to_flask()
    arm.move_to_pos(cfg.pos_dict['flask_contact_immersed'], **at_speed(dip_speed))
    arm.sample_in_flask = True


def unload_sample_at_speed(sample, lift_speed):
    # Same steps as arm.unload_sample, minus rinsing, with a chosen speed for the lift-out.
    # move_to_hover_pos does the real distance as an untimed relative move and only
    # applies **kwargs to a second, zero-distance move -- so the speed never took
    # effect there. Doing the lift as one real absolute move fixes that.
    immersed = cfg.pos_dict['flask_contact_immersed']
    above_flask = list(immersed[:2]) + [immersed[2] + cfg.hover_offset_dict['flask']] + list(immersed[3:])
    arm.move_to_pos(above_flask, **at_speed(lift_speed))
    arm.sample_in_flask = False
    arm.move_to_mid_station()
    arm.put_sample_back(sample)


# Print the plan, then wait for confirmation before moving anything.
print('Samples to run:', SAMPLES)
for sample in SAMPLES:
    dip, hold, lift = sample_settings(sample)
    at_speed(dip)   # raises if out of range, before anything moves
    at_speed(lift)
    print(f'  Sample {sample}: descend {dip} mm/s, hold {hold} s, lift {lift} mm/s')
if input('Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')

for sample in SAMPLES:
    dip, hold, lift = sample_settings(sample)

    print(f'[Sample {sample}] load (pick up, move to flask, descend at {dip} mm/s)')
    load_sample_at_speed(sample, dip)

    if hold > 0:
        print(f'[Sample {sample}] hold {hold} s')
        time.sleep(hold)

    print(f'[Sample {sample}] unload (lift at {lift} mm/s, put back, no rinsing)')
    unload_sample_at_speed(sample, lift)

print('Done.')

## Dip one sample, then leave it on the imaging rack

The same pick-up, dip, hold and lift as the immersion sweep above (through `mid_station`, nothing skipped). But instead of putting the sample back in its own slot, the arm then:

1. goes straight up from `mid_station` and crosses high above the bench, turning the gripper a quarter turn on the way;
2. comes down above the turned imaging rack, then slowly into slot 0 (the spot found with `imaging_quick_check.ipynb`), with the coated end on the left;
3. opens the gripper, goes back up, crosses back turning back, and waits at `mid_station`.

The sample's own slot stays empty. The coated end hangs off the left of the imaging rack, so the wet film touches nothing.

The release height is the usual grab height of the racks (z = 190.6). If the grab test in the quick check showed a better height, change `IMAGING_SPOT`.

**Before running:** the imaging rack's slot 0 is empty, the camera and its boom are clear of the gripper's way down to the imaging rack, the red wire on the gripper has slack, and your hand is on the e-stop. The first time, leave `CONFIRM_BEFORE_PLACING = True`: the arm stops 10 cm above the imaging rack and waits for your `yes`.

In [ ]:
# ===== Dip one sample, then leave it on the imaging rack (not back in its own slot) =====
# Pick up -> mid_station -> above the flask -> dip -> hold -> lift -> mid_station   (as in the sweep above)
# -> straight up -> across, high above the bench, turning a quarter turn -> down above the imaging rack
# -> slowly into slot 0 -> open the gripper -> back up -> across, turning back -> mid_station.
# Runs on its own: it does not need the immersion sweep cell above to be run first.
import time

SAMPLE = 0                    # which sample to dip (its usual rack slot)
DIP_SPEED_MM_S = 20.0         # speed going into the liquid
HOLD_S = 1                    # seconds to stay immersed
LIFT_SPEED_MM_S = 20.0        # speed coming out of the liquid

# Imaging rack slot 0 (rack turned, coated end on the left), found with imaging_quick_check.ipynb.
# The gripper lets go of the electrode here. z = 190.6 is the usual grab height of the racks.
IMAGING_SPOT = [-236.0, 332.1, 190.6, 180.0, 0.0, 0.0]
SAFE_Z = 500.0                # travel height between mid_station and the imaging rack
HOVER_MM = 100.0              # how high above the imaging spot the slow last part starts
PLACE_SPEED_MM_S = 10.0       # speed of that slow last part
CONFIRM_BEFORE_PLACING = True # stop 10 cm above the imaging rack and wait for your yes

cfg = arm.config


def at_speed(mm_s):
    """Default motion settings with the speed swapped to mm_s."""
    if not 0 < mm_s <= 100:
        raise ValueError(f'Speed {mm_s} mm/s must be between 0 and 100')
    return {**cfg.pos_settings_dict['default'], 'speed': mm_s}


def turned_from(rpy_a, rpy_b):
    """Largest angle (degrees) between two orientations given as roll, pitch, yaw."""
    return max(abs(((a - b) + 180) % 360 - 180) for a, b in zip(rpy_a, rpy_b))


mid = list(cfg.pos_dict['mid_station'])
immersed = cfg.pos_dict['flask_contact_immersed']
above_flask = list(immersed[:2]) + [immersed[2] + cfg.hover_offset_dict['flask']] + list(immersed[3:])
img_x, img_y, img_z = IMAGING_SPOT[:3]
img_rpy = IMAGING_SPOT[3:]

# The arm must point straight down (either the normal way or turned) and be at least 5 cm above
# the racks, so the first move can be straight up.
here = arm.get_cartesian_pos()
pointing_down = min(turned_from(here[3:], mid[3:]), turned_from(here[3:], img_rpy)) <= 5
if not pointing_down or here[2] < 240:
    raise SystemExit('The gripper must point straight down and be at least 5 cm above the racks (z >= 240) '
                     'before this cell. The arm did not move.')
at_mid = max(abs(a - b) for a, b in zip(here[:3], mid[:3])) <= 5 and turned_from(here[3:], mid[3:]) <= 5

route_to_mid = [] if at_mid else [
    ('straight up from where the arm is', [here[0], here[1], SAFE_Z, *here[3:]], 30),
    ('across to above mid_station', [mid[0], mid[1], SAFE_Z, *mid[3:]], 30),
    ('down to mid_station', mid, 30),
]
route_to_imaging = [
    ('straight up from mid_station', [mid[0], mid[1], SAFE_Z, *mid[3:]], 30),
    ('across, turning a quarter turn', [img_x, img_y, SAFE_Z, *img_rpy], 30),
    ('down to 10 cm above the imaging rack', [img_x, img_y, img_z + HOVER_MM, *img_rpy], 20),
]
place = ('slowly into imaging slot 0', [img_x, img_y, img_z, *img_rpy], PLACE_SPEED_MM_S)
route_back = [
    ('up to 10 cm above the imaging rack', [img_x, img_y, img_z + HOVER_MM, *img_rpy], 20),
    ('straight up', [img_x, img_y, SAFE_Z, *img_rpy], 30),
    ('across, turning back', [mid[0], mid[1], SAFE_Z, *mid[3:]], 30),
    ('down to mid_station', mid, 30),
]


def go(steps):
    for name, pose, speed in steps:
        print('   ->', name)
        arm.move_to_pos(pose, **at_speed(speed))


# Print the plan, then wait for confirmation before moving anything.
for speed in (DIP_SPEED_MM_S, LIFT_SPEED_MM_S, PLACE_SPEED_MM_S):
    at_speed(speed)           # raises if out of range, before anything moves
print(f'Sample {SAMPLE}: dip {DIP_SPEED_MM_S} mm/s, hold {HOLD_S} s, lift {LIFT_SPEED_MM_S} mm/s,')
print(f'then leave it on the imaging rack at {[round(v, 1) for v in IMAGING_SPOT]}.')
for name, pose, speed in route_to_mid:
    print(f'  first: {name:40s} -> {[round(v, 1) for v in pose]} at {speed} mm/s')
print('Check: imaging slot 0 empty, camera clear of the way down, red wire has slack, hand on the e-stop.')
if input('Type yes to start: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')

arm.assert_ready()
if route_to_mid:
    print('[0] to mid_station, high above the bench')
    go(route_to_mid)

print(f'[1] pick up sample {SAMPLE}, then to the flask')
arm.pick_up_sample(SAMPLE)
arm.move_to_mid_station()
arm.move_to_flask()

print(f'[2] dip at {DIP_SPEED_MM_S} mm/s')
arm.move_to_pos(immersed, **at_speed(DIP_SPEED_MM_S))
arm.sample_in_flask = True
if HOLD_S > 0:
    print(f'[3] hold {HOLD_S} s')
    time.sleep(HOLD_S)

print(f'[4] lift at {LIFT_SPEED_MM_S} mm/s, then to mid_station')
arm.move_to_pos(above_flask, **at_speed(LIFT_SPEED_MM_S))
arm.sample_in_flask = False
arm.move_to_mid_station()

print('[5] to the imaging rack')
go(route_to_imaging)
if CONFIRM_BEFORE_PLACING and input('Gripper 10 cm above imaging slot 0, coated end on the left, nothing in the way? '
                                    'Type yes to put the sample down: ').strip().lower() != 'yes':
    raise SystemExit('Stopped 10 cm above the imaging rack, still holding the sample. '
                     'Run the next cell to put it back in its own slot.')
go([place])
arm.open_gripper()
print('[6] sample left on the imaging rack; back to mid_station')
go(route_back)
print(f'Done. Sample {SAMPLE} is on the imaging rack, slot 0. Its own slot is empty.')

In [ ]:
# ===== Only if the cell above stopped 10 cm above the imaging rack: put the sample back in its own slot =====
# Straight up -> across, high above the bench, turning back -> mid_station -> into its own slot (as usual).
here = arm.get_cartesian_pos()
if max(abs(here[0] - img_x), abs(here[1] - img_y), abs(here[2] - (img_z + HOVER_MM))) > 5:
    raise SystemExit('The arm is not 10 cm above the imaging rack, so this cell does nothing.')
if input(f'Type yes to take sample {SAMPLE} back to its own slot: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
go([
    ('straight up', [img_x, img_y, SAFE_Z, *img_rpy], 30),
    ('across, turning back', [mid[0], mid[1], SAFE_Z, *mid[3:]], 30),
    ('down to mid_station', mid, 30),
])
arm.put_sample_back(SAMPLE)
print(f'Sample {SAMPLE} is back in its own slot.')

## Dip one sample, then leave it on the RIGHT imaging rack

The same pick-up, dip, hold and lift as the immersion sweep above (through `mid_station`, nothing skipped). Then, instead of putting the sample back in its own slot, the arm takes it to the right imaging rack behind the sample racks, where it lies left to right with the coated end on the right, under the camera:

1. straight up from `mid_station` and across high above the bench at 100 mm/s, turning the gripper a quarter turn;
2. down beside the imaging spot, 7 cm to the left (away from the camera), where it stops to ask: `yes` puts the sample down, `back` takes it back to its own slot instead;
3. low over the rack, slides the electrode in under the camera, lowers it until it just rests on the rack (the height taught with `imaging_right.ipynb`), lets go slowly (so the electrode does not roll and stays face up), lifts straight up, slides back out and returns to `mid_station`.

The sample's own slot stays empty until the last cell takes it back. The moves are in `robotic_testing/imaging_rack.py`.

**Before running:** the imaging slot is empty, the sample is in its own slot, the camera lens is at least 5 cm above the rack, the red wire on the gripper has slack, and your hand is on the e-stop. Run the setup cell first (nothing moves there), then the dip cell.

In [ ]:
# ===== Right imaging rack: where it is and how the arm moves there. Nothing moves in this cell. =====
# x, y and z were taught with imaging_right.ipynb (teach cell), holding a real sample, coated end on the right,
# under the camera. At z = 202.2 the electrode just rests on the rack, flat, when the gripper lets go: lower
# (195.6 or 190.6) pushes its metal end down, so the coated end tips up and it rolls when let go.
from robotic_testing.imaging_rack import ImagingRack

right_rack = ImagingRack(
    arm,
    spot=[299.0, 488.6, 202.2],     # x, y, z where the gripper lets go of the electrode
    coated_end='right',             # coated end (and camera) on the right, metal end on the left
    travel_speed=100.0,             # mm/s after the dip: up, across high above the bench, and down beside the rack
    slide_speed=30.0,               # mm/s low over the rack, sliding in under the camera
    place_speed=10.0,               # mm/s for the last 2.5 cm into the slot
    confirm_before_placing=True,    # stop above the rack and wait for your yes (the wait adds drying time)
    release_gripper_speed=1000,     # how fast the fingers open when letting go: 1000 = slowest (the usual is 5000)
)
print(right_rack.describe())

In [ ]:
# ===== Dip one sample, then leave it on the right imaging rack (not back in its own slot) =====
# Pick up -> mid_station -> above the flask -> dip -> hold -> lift -> mid_station   (as in the sweep above)
# -> high across, turning -> down beside the imaging spot (stops to ask: yes, or back to its own slot)
# -> low over the rack, slide in under the camera -> put down -> let go -> slide back out -> mid_station.
right_rack.dip_and_leave(
    sample=0,          # which sample to dip (its usual rack slot)
    dip_speed=20.0,    # mm/s into the liquid
    hold_s=1,          # seconds immersed
    lift_speed=20.0,   # mm/s out of the liquid
)

### After imaging: take the sample back to its own slot

In [ ]:
# ===== After imaging: take the sample from the right imaging rack back to its own slot =====
# mid_station -> high across, turning -> down beside the imaging spot -> open the gripper -> low over the rack
# -> slide in -> down -> close on the electrode -> up a little -> slide out -> high across, turning back
# -> mid_station -> into its own slot (as usual). If the gripper closes on nothing, it comes back empty.
right_rack.take_back(sample=0)

## Free-form motion

Held to the envelope in the arm's config. A move outside it, or bigger than one step, is
refused before anything moves -- `dry_run=True` reports where a move would end up without
going there.

The envelope is a box with a keep-out cylinder around the base. It constrains
`move_relative` and friends; it does **not** constrain `home()` or any other joint-space
move.

In [ ]:
from robotic_testing.common.robotic_arms.xarm_control import SafetyError

# where would this end up?
print('predicted:', arm.move_relative(dx=30, dy=-20, dz=50, dry_run=True))

# refused before it starts
try:
    arm.move_relative(dz=5000)
except SafetyError as error:
    print('refused: ', error)

# actually move
# arm.move_relative(dx=30, dy=-20, dz=50)
# arm.move_up(40)
# arm.move_down(40)
# arm.move_to_pose(z=300)          # absolute; None keeps the current value
# arm.move_relative_path([[0, 0, 80], [60, 0, 0], [0, 0, -80]], dry_run=True)

## Stopping, and recovering

`emergency_stop()` puts the controller into the stop state, abandoning the current move.
`reset_safety_state()` clears the fault and re-enables motion -- only once the cause is
understood.

In [ ]:
# arm.emergency_stop()
# arm.reset_safety_state()